# 第三章 Notebook 2：声音合成入门

本 Notebook 介绍用振荡器、滤波器和包络生成声音的基本方法。

Notebook 1（`01_midi_rendering_basics.ipynb`）采用 FluidSynth + SoundFont 的采样音源工作流。这里直接用数学表达式生成波形，不读取预录制的乐器音频。

内容：
1. **工具函数**：包络、归一化、滤波等基础组件
2. **加法合成**：叠加整数倍频正弦分量，比较谐波数量与权重
3. **减法合成**：从宽频谱源中滤除频率分量，比较低通、高通和带通结果
4. **相位调制**：分析调制指数与频率比对边带的影响
5. **ADSR 包络**：比较包络参数对听感的影响
6. **正弦波与三种合成方式总览**：波形与频谱对比

依赖：`numpy`、`soundfile`、`matplotlib` 和 `IPython.display.Audio`

---

**前置知识检查**

本 Notebook 涉及以下内容。若不熟悉这些概念，建议先浏览相关资料：
- 三角函数（正弦波 `sin(2πft)`）的基本含义
- 频谱、频率、谐波的基础概念（不需要深入信号处理）
- `numpy` 数组操作和 `matplotlib` 绘图基础

> 所有示例均为单音、固定采样率的局部实验；听感描述只对应所列参数和峰值归一化后的输出。


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display, Markdown

plt.rcParams["font.sans-serif"] = [
    "PingFang SC", "Hiragino Sans GB", "Microsoft YaHei",
    "SimHei", "Arial Unicode MS", "Noto Sans CJK SC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
for _k in ("figure.facecolor", "axes.facecolor"):
    plt.rcParams[_k] = "white"
for _k in ("axes.edgecolor", "axes.labelcolor", "xtick.color", "ytick.color", "text.color"):
    plt.rcParams[_k] = "black"

_p = os.getcwd()
while not os.path.exists(os.path.join(_p, "CODE", "datasets")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError("未找到包含 CODE/datasets 的项目根目录")
    _p = _parent
BASE_DIR = _p
NOTEBOOK_DIR = os.path.join(BASE_DIR, "CODE", "chapter03")
FIGURES_DIR = os.path.join(NOTEBOOK_DIR, "output_figures")
AUDIO_DIR = os.path.join(NOTEBOOK_DIR, "output_audio")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

SR = 44100
DUR = 1.5
F0 = 440.0
NOTE_OFF = 1.0
t = np.arange(int(SR * DUR)) / SR

def project_path(path):
    return os.path.relpath(path, BASE_DIR)


def notebook_path(path):
    return os.path.relpath(path, NOTEBOOK_DIR)


print(f"采样率：{SR} Hz；信号时长：{DUR} s；基频参数：{F0} Hz")
print(f"Note Off：{NOTE_OFF} s；总采样点数：{len(t)}")

## 1. 工具函数

以下函数用于各项实验：
- `normalize`：将峰值统一到指定幅度；它不等同于响度归一化，也不保证感知音量相同
- `adsr_envelope`：根据显式 Note Off 时刻生成起音、衰减、保持和释放包络
- `fft_filter`：对整段有限长信号施加理想矩形频率掩膜

In [ ]:
def normalize(audio_data, peak=0.75):
    """峰值归一化。"""
    mx = np.max(np.abs(audio_data))
    return audio_data / mx * peak if mx > 0 else audio_data


def adsr_envelope(time_arr, note_off, attack=0.03, decay=0.05, sustain_level=0.7, release=0.2):
    """生成线性 ADSR 包络；release 从 note_off 时刻的当前电平开始。"""
    if attack < 0 or decay < 0 or release < 0:
        raise ValueError("ADSR 时间参数不能为负数")
    if not 0 <= sustain_level <= 1:
        raise ValueError("sustain_level 必须在 0 到 1 之间")
    if not time_arr[0] <= note_off <= time_arr[-1]:
        raise ValueError("note_off 必须位于时间数组范围内")

    envelope = np.zeros_like(time_arr)
    a_mask = (time_arr < note_off) & (time_arr < attack)
    envelope[a_mask] = time_arr[a_mask] / attack if attack > 0 else 1.0

    d_mask = (time_arr < note_off) & (time_arr >= attack) & (time_arr < attack + decay)
    if decay > 0:
        envelope[d_mask] = 1.0 - (1.0 - sustain_level) * (time_arr[d_mask] - attack) / decay

    s_mask = (time_arr < note_off) & (time_arr >= attack + decay)
    envelope[s_mask] = sustain_level

    if note_off < attack and attack > 0:
        release_start_level = note_off / attack
    elif note_off < attack + decay and decay > 0:
        release_start_level = 1.0 - (1.0 - sustain_level) * (note_off - attack) / decay
    else:
        release_start_level = sustain_level
    r_mask = time_arr >= note_off
    if release > 0:
        envelope[r_mask] = release_start_level * np.clip(
            1.0 - (time_arr[r_mask] - note_off) / release, 0.0, 1.0
        )
    return envelope


def fft_filter(audio_data, sr, lowpass_hz=None, highpass_hz=None):
    """以 FFT 矩形掩膜实现离线理想低通、高通或带通。"""
    spec = np.fft.rfft(audio_data)
    freq_vals = np.fft.rfftfreq(len(audio_data), d=1 / sr)
    if lowpass_hz is not None:
        spec[freq_vals > lowpass_hz] = 0
    if highpass_hz is not None:
        spec[freq_vals < highpass_hz] = 0
    return np.fft.irfft(spec, n=len(audio_data))


def plot_wave_and_spectrum(audio_data, sr, title, color="#2c3e50", spec_color="#8e44ad",
                           zoom_ms=25, freq_max=5000):
    """绘制波形局部放大 + 频谱。"""
    fig_out, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
    zoom_samples = int(zoom_ms / 1000 * sr)
    t_ms = np.arange(zoom_samples) / sr * 1000
    ax1.plot(t_ms, audio_data[:zoom_samples], color=color, linewidth=0.8)
    ax1.set_xlabel("时间（ms）")
    ax1.set_ylabel("振幅")
    ax1.set_title(f"{title} — 波形")
    ax1.set_ylim(-1, 1)

    win = np.hanning(len(audio_data))
    spec = np.abs(np.fft.rfft(audio_data * win))
    freq_vals = np.fft.rfftfreq(len(audio_data), d=1 / sr)
    spec_db = 20 * np.log10(spec / (spec.max() + 1e-12) + 1e-6)
    freq_mask = freq_vals <= freq_max
    ax2.plot(freq_vals[freq_mask], spec_db[freq_mask], color=spec_color, linewidth=0.8)
    ax2.set_xlabel("频率（Hz）")
    ax2.set_ylabel("相对幅度（dB）")
    ax2.set_title(f"{title} — 频谱")
    ax2.set_ylim(-80, 5)
    plt.tight_layout()
    return fig_out


def play(audio_data, sr=44100, name=""):
    """保存并播放音频。"""
    if name:
        path = os.path.join(AUDIO_DIR, name)
        sf.write(path, audio_data.astype(np.float32), sr)
        display(Audio(url=notebook_path(path)))
    else:
        display(Audio(audio_data, rate=sr))

print("工具函数就绪")

## 2. 加法合成：谐波叠加

加法合成把多个正弦分量相加。本节只使用 440 Hz 的整数倍频，因此这些分量称为谐波；一般加法合成也可以使用非整数倍频的分音。

以下单音均含 440 Hz 基频，谐波数量、幅度和包络共同影响频谱与听感。基频是本例音高知觉的重要线索，但实际音高知觉还受谱结构、包络和听觉条件影响。

In [ ]:
env = adsr_envelope(t, NOTE_OFF, attack=0.02, decay=0.05, sustain_level=0.8, release=0.25)

# 基础加法合成：5 个谐波
harmonics_basic = [(1, 1.0), (2, 0.45), (3, 0.25), (4, 0.15), (5, 0.08)]
additive_basic = normalize(
    sum((w * np.sin(2 * np.pi * F0 * h * t) for h, w in harmonics_basic), start=np.zeros_like(t)) * env
)

fig = plot_wave_and_spectrum(additive_basic, SR, "加法合成 (5 谐波)")
plt.show()
play(additive_basic, SR, "synth_additive.wav")

### 谐波数量对频谱和听感的影响

在本例固定的 $1/h^{0.7}$ 权重下，增加谐波数量会加入更多高频能量。以下比较 1、3、8、20 个谐波；各信号仅做峰值归一化。

In [ ]:
harmonic_counts = [1, 3, 8, 20]
fig, axes = plt.subplots(2, len(harmonic_counts), figsize=(16, 6))

for col, n_harm in enumerate(harmonic_counts):
    weights = [1.0 / (h ** 0.7) for h in range(1, n_harm + 1)]
    audio = normalize(
        sum((w * np.sin(2 * np.pi * F0 * h * t) for h, w in zip(range(1, n_harm + 1), weights)), start=np.zeros_like(t)) * env
    )

    zoom_n = int(0.025 * SR)
    axes[0, col].plot(np.arange(zoom_n) / SR * 1000, audio[:zoom_n], color="#2c3e50", linewidth=0.8)
    axes[0, col].set_title(f"{n_harm} 个谐波")
    axes[0, col].set_ylim(-1, 1)
    axes[0, col].set_xlabel("时间（ms）")
    if col == 0:
        axes[0, col].set_ylabel("振幅")

    window = np.hanning(len(audio))
    spectrum = np.abs(np.fft.rfft(audio * window))
    freqs = np.fft.rfftfreq(len(audio), d=1 / SR)
    spectrum_db = 20 * np.log10(spectrum / (spectrum.max() + 1e-12) + 1e-6)
    mask = freqs <= 5000
    axes[1, col].plot(freqs[mask], spectrum_db[mask], color="#8e44ad", linewidth=0.8)
    axes[1, col].set_ylim(-80, 5)
    axes[1, col].set_xlabel("频率（Hz）")
    if col == 0:
        axes[1, col].set_ylabel("相对幅度（dB）")

    fname = f"synth_additive_{n_harm}h.wav"
    sf.write(os.path.join(AUDIO_DIR, fname), audio.astype(np.float32), SR)

fig.suptitle("加法合成：谐波数量对波形与频谱的影响", y=1.02)
plt.tight_layout()
plt.show()

for n_harm in harmonic_counts:
    display(Markdown(f"**{n_harm} 个谐波**"))
    display(Audio(url=notebook_path(os.path.join(AUDIO_DIR, f"synth_additive_{n_harm}h.wav"))))

### 谐波权重：比较有限谐波级数

以下三种权重都只叠加前 15 个谐波。$1/n$ 的全谐波级数接近锯齿波，奇次 $1/n$ 级数接近方波；有限项截断和统一包络使结果不同于理想周期波。

In [ ]:
decay_profiles = {
    "全谐波 1/n": lambda h: 1.0 / h,
    "全谐波 1/n^2": lambda h: 1.0 / (h ** 2),
    "奇次谐波 1/n": lambda h: (1.0 / h) if h % 2 == 1 else 0.0,
}

n_harmonics = 15
fig, axes = plt.subplots(1, len(decay_profiles), figsize=(15, 3.5))

for col, (label, weight_fn) in enumerate(decay_profiles.items()):
    audio = normalize(
        sum((weight_fn(h) * np.sin(2 * np.pi * F0 * h * t) for h in range(1, n_harmonics + 1)), start=np.zeros_like(t)) * env
    )
    zoom_n = int(0.025 * SR)
    axes[col].plot(np.arange(zoom_n) / SR * 1000, audio[:zoom_n], color="#2c3e50", linewidth=0.8)
    axes[col].set_title(label)
    axes[col].set_xlabel("时间（ms）")
    axes[col].set_ylim(-1, 1)
    if col == 0:
        axes[col].set_ylabel("振幅")

    safe_name = label.split("(")[0].strip().replace("/", "_").replace(" ", "_")
    fname = f"synth_additive_{safe_name}.wav"
    sf.write(os.path.join(AUDIO_DIR, fname), audio.astype(np.float32), SR)

plt.suptitle("不同谐波权重下的波形", y=1.02)
plt.tight_layout()
plt.show()

for label in decay_profiles:
    safe_name = label.split("(")[0].strip().replace("/", "_").replace(" ", "_")
    display(Markdown(f"**{label}**"))
    display(Audio(url=notebook_path(os.path.join(AUDIO_DIR, f"synth_additive_{safe_name}.wav"))))

## 3. 减法合成：从宽频谱源中选取频段

减法合成先生成含较多频率分量的源信号，再用滤波器衰减或去除部分分量。本例用前 34 个谐波近似锯齿波。

代码直接把截止频率外的 FFT 频点置零，相当于对整段信号施加理想、零相位、砖墙式频率掩膜。该操作不是实时滤波器，边界不连续时可能出现时域振铃；图和音频只反映频段选择结果，不代表实际合成器滤波器的瞬态响应。

In [ ]:
# 源波形：前 34 个谐波叠加（近似锯齿波）
rich = sum((np.sin(2 * np.pi * F0 * h * t) / h for h in range(1, 35)), start=np.zeros_like(t))

filters = {
    "原始 (无滤波)":   lambda x: x,
    "低通 1600 Hz":    lambda x: fft_filter(x, SR, lowpass_hz=1600),
    "低通 800 Hz":     lambda x: fft_filter(x, SR, lowpass_hz=800),
    "高通 1000 Hz":    lambda x: fft_filter(x, SR, highpass_hz=1000),
    "带通 800-2000 Hz": lambda x: fft_filter(x, SR, lowpass_hz=2000, highpass_hz=800),
}

fig, axes = plt.subplots(2, len(filters), figsize=(18, 6))

for col, (label, filt_fn) in enumerate(filters.items()):
    filtered = normalize(filt_fn(rich) * env)
    zoom_n = int(0.025 * SR)
    axes[0, col].plot(np.arange(zoom_n) / SR * 1000, filtered[:zoom_n], color="#2c3e50", linewidth=0.8)
    axes[0, col].set_title(label, fontsize=9)
    axes[0, col].set_ylim(-1, 1)
    axes[0, col].set_xlabel("时间（ms）")
    if col == 0:
        axes[0, col].set_ylabel("振幅")

    window = np.hanning(len(filtered))
    spectrum = np.abs(np.fft.rfft(filtered * window))
    freqs = np.fft.rfftfreq(len(filtered), d=1 / SR)
    spectrum_db = 20 * np.log10(spectrum / (spectrum.max() + 1e-12) + 1e-6)
    mask = freqs <= 5000
    axes[1, col].plot(freqs[mask], spectrum_db[mask], color="#e67e22", linewidth=0.8)
    axes[1, col].set_ylim(-80, 5)
    axes[1, col].set_xlabel("频率（Hz）")
    if col == 0:
        axes[1, col].set_ylabel("相对幅度（dB）")

    safe = label.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
    sf.write(os.path.join(AUDIO_DIR, f"synth_sub_{safe}.wav"), filtered.astype(np.float32), SR)

fig.suptitle("减法合成：不同滤波器对同一源波形的效果", y=1.02)
plt.tight_layout()
plt.show()

for label in filters:
    safe = label.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "")
    display(Markdown(f"**{label}**"))
    display(Audio(url=notebook_path(os.path.join(AUDIO_DIR, f"synth_sub_{safe}.wav"))))

## 4. 相位调制及其边带

本节直接把正弦调制信号加到载波相位：

`y(t) = sin(2π·f_c·t + β·sin(2π·f_m·t))`

其中 $f_c$ 是载波频率，$f_m$ 是调制器频率，$β$ 是相位调制指数。该表达式实现的是相位调制（PM）；在正弦、定频调制条件下，它与常见数字 FM 形式具有对应关系，但不能把任意相位调制与频率调制视为同一算法。频谱边带位于 $|f_c \pm kf_m|$，其幅度随 $β$ 按贝塞尔函数重新分配。

In [ ]:
# 基础相位调制
carrier_freq = F0
mod_freq = F0 * 2  # 频率比 1:2
mod_index = 4.0

pm_basic = normalize(
    np.sin(2 * np.pi * carrier_freq * t + mod_index * np.sin(2 * np.pi * mod_freq * t)) * env
)

fig = plot_wave_and_spectrum(pm_basic, SR, "相位调制 (β=4, fc:fm=1:2)")
plt.show()
play(pm_basic, SR, "synth_pm.wav")

### 调制指数的影响

在固定 $f_c:f_m=1:2$ 时，增大 $β$ 通常会扩大具有显著能量的边带范围，但各边带幅度并不单调增加。以下比较四个参数值。

In [ ]:
mod_indices = [0.5, 2.0, 5.0, 8.0]
fig, axes = plt.subplots(2, len(mod_indices), figsize=(16, 6))

for col, beta in enumerate(mod_indices):
    audio = normalize(
        np.sin(2 * np.pi * F0 * t + beta * np.sin(2 * np.pi * (F0 * 2) * t)) * env
    )
    zoom_n = int(0.025 * SR)
    axes[0, col].plot(np.arange(zoom_n) / SR * 1000, audio[:zoom_n], color="#2c3e50", linewidth=0.8)
    axes[0, col].set_title(f"β = {beta}")
    axes[0, col].set_ylim(-1, 1)
    axes[0, col].set_xlabel("时间（ms）")
    if col == 0:
        axes[0, col].set_ylabel("振幅")

    window = np.hanning(len(audio))
    spectrum = np.abs(np.fft.rfft(audio * window))
    freqs = np.fft.rfftfreq(len(audio), d=1 / SR)
    spectrum_db = 20 * np.log10(spectrum / (spectrum.max() + 1e-12) + 1e-6)
    mask = freqs <= 5000
    axes[1, col].plot(freqs[mask], spectrum_db[mask], color="#e74c3c", linewidth=0.8)
    axes[1, col].set_ylim(-80, 5)
    axes[1, col].set_xlabel("频率（Hz）")
    if col == 0:
        axes[1, col].set_ylabel("相对幅度（dB）")

    sf.write(os.path.join(AUDIO_DIR, f"synth_pm_beta{beta}.wav"), audio.astype(np.float32), SR)

fig.suptitle("相位调制：调制指数 β 对频谱的影响", y=1.02)
plt.tight_layout()
plt.show()

for beta in mod_indices:
    display(Markdown(f"**β = {beta}**"))
    display(Audio(url=notebook_path(os.path.join(AUDIO_DIR, f"synth_pm_beta{beta}.wav"))))

### 频率比：边带频率网格

当 $f_c$ 与 $f_m$ 之比为有理数时，边带落在某个共同基频的整数倍网格上；整数比是其中的特例。相对于 440 Hz 载波，$1:1$ 和 $1:2$ 的边带位于 440 Hz 的整数倍，$1:1.5$ 位于 220 Hz 网格。理想无理数比不会形成单一谐波网格；计算机中的浮点数只是在有限精度下近似该条件。听感还取决于调制指数、包络、音区和播放系统。

In [ ]:
freq_ratios = {
    "1:1 (440 Hz 网格)": 1.0,
    "1:2 (440 Hz 网格)": 2.0,
    "1:1.5 (220 Hz 网格)": 1.5,
    "1:√2 (近似非谐波)": np.sqrt(2),
}

fig, axes = plt.subplots(2, len(freq_ratios), figsize=(16, 6))
beta = 4.0

for col, (label, ratio) in enumerate(freq_ratios.items()):
    audio = normalize(
        np.sin(2 * np.pi * F0 * t + beta * np.sin(2 * np.pi * (F0 * ratio) * t)) * env
    )
    zoom_n = int(0.025 * SR)
    axes[0, col].plot(np.arange(zoom_n) / SR * 1000, audio[:zoom_n], color="#2c3e50", linewidth=0.8)
    axes[0, col].set_title(label, fontsize=10)
    axes[0, col].set_ylim(-1, 1)
    axes[0, col].set_xlabel("时间（ms）")
    if col == 0:
        axes[0, col].set_ylabel("振幅")

    window = np.hanning(len(audio))
    spectrum = np.abs(np.fft.rfft(audio * window))
    freqs = np.fft.rfftfreq(len(audio), d=1 / SR)
    spectrum_db = 20 * np.log10(spectrum / (spectrum.max() + 1e-12) + 1e-6)
    mask = freqs <= 5000
    axes[1, col].plot(freqs[mask], spectrum_db[mask], color="#e74c3c", linewidth=0.8)
    axes[1, col].set_ylim(-80, 5)
    axes[1, col].set_xlabel("频率（Hz）")
    if col == 0:
        axes[1, col].set_ylabel("相对幅度（dB）")

    safe = label.split("(")[0].strip().replace(":", "_").replace(".", "p")
    sf.write(os.path.join(AUDIO_DIR, f"synth_pm_ratio_{safe}.wav"), audio.astype(np.float32), SR)

fig.suptitle("相位调制：频率比对频谱的影响（β=4）", y=1.02)
plt.tight_layout()
plt.show()

for label in freq_ratios:
    safe = label.split("(")[0].strip().replace(":", "_").replace(".", "p")
    display(Markdown(f"**{label}**"))
    display(Audio(url=notebook_path(os.path.join(AUDIO_DIR, f"synth_pm_ratio_{safe}.wav"))))

## 5. ADSR 包络：时间塑造听感

ADSR 的 Attack、Decay 和 Sustain 描述 Note On 之后的包络，Release 从 Note Off 时刻开始。本节把 Note Off 固定在 1.0 秒，并对同一谐波信号使用四组线性包络参数。各名称概括主要包络参数，并非乐器的声学模型。

In [ ]:
adsr_presets = {
    "短起音、快衰减": {"filename": "synth_adsr_钢琴感.wav", "attack": 0.005, "decay": 0.15, "sustain_level": 0.3, "release": 0.3},
    "长起音、高保持": {"filename": "synth_adsr_弦乐感.wav", "attack": 0.25, "decay": 0.1, "sustain_level": 0.85, "release": 0.4},
    "短起音、满保持": {"filename": "synth_adsr_管风琴感.wav", "attack": 0.02, "decay": 0.01, "sustain_level": 1.0, "release": 0.05},
    "极短起音、无保持": {"filename": "synth_adsr_拨弦感.wav", "attack": 0.002, "decay": 0.3, "sustain_level": 0.0, "release": 0.1},
}

base_tone = sum(
    ((1.0 / h) * np.sin(2 * np.pi * F0 * h * t) for h in range(1, 12)), start=np.zeros_like(t)
)

fig, axes = plt.subplots(2, len(adsr_presets), figsize=(16, 5))

for col, (label, params) in enumerate(adsr_presets.items()):
    env_params = {key: value for key, value in params.items() if key != "filename"}
    env_curve = adsr_envelope(t, NOTE_OFF, **env_params)
    audio = normalize(base_tone * env_curve)

    # 包络曲线
    axes[0, col].plot(t, env_curve, color="#27ae60", linewidth=1.5)
    axes[0, col].set_title(label, fontsize=9)
    axes[0, col].set_ylim(-0.05, 1.1)
    axes[0, col].set_xlabel("时间（s）")
    if col == 0:
        axes[0, col].set_ylabel("包络幅度")

    # 波形
    axes[1, col].plot(t, audio, color="#2c3e50", linewidth=0.3)
    axes[1, col].set_ylim(-1, 1)
    axes[1, col].set_xlabel("时间（s）")
    if col == 0:
        axes[1, col].set_ylabel("振幅")

    sf.write(os.path.join(AUDIO_DIR, params["filename"]), audio.astype(np.float32), SR)

fig.suptitle("ADSR 包络对同一谐波信号的影响", y=1.02)
plt.tight_layout()
plt.show()

for label, params in adsr_presets.items():
    display(Markdown(f"**{label}**"))
    display(Audio(url=notebook_path(os.path.join(AUDIO_DIR, params["filename"]))))

## 6. 正弦波与三种合成方式总览

下图比较正弦波基准、加法合成、理想频率掩膜减法合成和相位调制。四个信号使用相同的 440 Hz 基频参数和线性 ADSR 参数，峰值幅度均归一化为 0.75；上排显示起音后 25 ms 波形，下排显示加 Hann 窗后的相对幅度谱。

In [ ]:
env_overview = adsr_envelope(t, NOTE_OFF, attack=0.03, decay=0.05, sustain_level=0.7, release=0.18)

sine = np.sin(2 * np.pi * F0 * t) * env_overview

additive = sum(
    (weight * np.sin(2 * np.pi * F0 * harmonic * t)
     for harmonic, weight in [(1, 1.0), (2, 0.45), (3, 0.25), (4, 0.15), (5, 0.08)]),
    start=np.zeros_like(t)
) * env_overview

rich_source = sum(
    (np.sin(2 * np.pi * F0 * harmonic * t) / harmonic
     for harmonic in range(1, 35)),
    start=np.zeros_like(t)
)
subtractive = fft_filter(rich_source, SR, lowpass_hz=1600) * env_overview

pm_overview = np.sin(
    2 * np.pi * F0 * t + 4.0 * np.sin(2 * np.pi * (F0 * 2) * t)
) * env_overview

SYNTH_OVERVIEW = {
    "正弦波基准": normalize(sine),
    "加法合成":   normalize(additive),
    "减法合成":   normalize(subtractive),
    "相位调制":   normalize(pm_overview),
}

fig, axes = plt.subplots(2, len(SYNTH_OVERVIEW), figsize=(16, 6.5))
_zoom_samples = int(0.025 * SR)

for col, (synth_name, sig) in enumerate(SYNTH_OVERVIEW.items()):
    axes[0, col].plot(np.arange(_zoom_samples) / SR * 1000, sig[:_zoom_samples],
                      color="#2c3e50", linewidth=0.8)
    axes[0, col].set_title(synth_name, fontsize=12)
    axes[0, col].set_xlabel("时间（ms）", fontsize=10)
    axes[0, col].set_ylim(-0.85, 0.85)
    if col == 0:
        axes[0, col].set_ylabel("振幅")

    _win = np.hanning(len(sig))
    _spec = np.abs(np.fft.rfft(sig * _win))
    _freq_vals = np.fft.rfftfreq(len(sig), d=1 / SR)
    _spec_db = 20 * np.log10(_spec / (_spec.max() + 1e-12) + 1e-6)
    _freq_mask = _freq_vals <= 5000
    axes[1, col].plot(_freq_vals[_freq_mask], _spec_db[_freq_mask], color="#8e44ad", linewidth=0.8)
    axes[1, col].set_xlabel("频率（Hz）", fontsize=10)
    axes[1, col].set_ylim(-80, 5)
    if col == 0:
        axes[1, col].set_ylabel("相对幅度（dB）")

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, "fig_synthesis_examples.png")
fig.savefig(out_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"已保存：{project_path(out_path)}")

## 小结

| 合成方式 | 核心思路 | 关键参数 | 音色特点 |
|----------|---------|---------|---------|
| 加法合成 | 叠加正弦分量 | 分音频率、幅度与相位 | 可直接控制谱分量，参数量随分量数增加 |
| 减法合成 | 从宽频谱源中衰减部分频率 | 源信号、滤波器与包络 | 本例只展示离线理想频率掩膜 |
| 相位调制 | 用调制器改变载波相位 | 调制指数、载波与调制器频率 | 少量参数可产生多组边带 |

本 Notebook 的三种算法实现不读取预录制乐器素材。TimGM6mb 预设则主要依赖采样及其区域与播放参数。输出差异同时受振荡器或采样内容、包络、滤波、归一化和渲染设置影响。
